# Домашняя работа: ASR для call-центра

## Контекст

Представим, что компания хочет улучшить качество распознавания речи операторов колл-центра с индийским акцентом. Вам необходимо исследовать, насколько хорошо базовая модель Whisper справляется с такой речью и помогает ли её дополнительное обучение на специализированном наборе данных.
Задача - сравнить качество исходной и дообученной моделей и проанализировать полученные результаты.

## Что вы будете делать

1. Возьмёте реальные записи индийского английского как обучающие данные
2. Сгенерируете свой собственный тестовый набор — 20 типичных call-центровых фраз, озвученных с индийским акцентом через TTS
3. Замерите baseline WER замороженной модели на этих звонках
4. Дообучите Whisper с LoRA на реальных индийских данных
5. Сравните до/после и разберёте конкретные ошибки

Ссылка для сдачи ДЗ: https://forms.gle/yPwBt3ZV3JikENaS7

In [1]:
# Установка зависимостей
#   transformers — Whisper и SpeechT5
#   peft — LoRA-адаптеры для эффективного файнтюнинга
#   torchaudio — ресемплинг аудио
#   soundfile  — чтение/запись wav
#   jiwer — расчёт WER и пословный анализ ошибок
#   accelerate — mixed precision и оптимизация обучения
#   speechbrain — извлечение speaker embeddings из реальных записей

!pip install -q transformers datasets peft torchaudio soundfile jiwer accelerate speechbrain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 55.0 MB/s eta 0:00:00


---
## 1. Загрузка данных - 1 балл

Загружаем реальные обучающие данные и генерируем синтетический тестовый набор.

Загружаем Svarah, Indian-accented English ASR dataset, который включает в себя 9.6 часов индийского английского, 117 дикторов, ~1GB.

Датасет `ai4bharat/Svarah` требует авторизации и принятия условий доступа на Hugging Face.

Перед запуском кода нужно:

1. Открыть страницу датасета `ai4bharat/Svarah` на Hugging Face.
2. Принять условия доступа к датасету.
3. Создать или скопировать свой Hugging Face access token.
4. Вставить токен в Colab через переменную окружения.

In [2]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass(TOKEN)


# После этого датасет можно загрузить:

# from datasets import load_dataset

# dataset = load_dataset(
#     "ai4bharat/Svarah",
#     split="test",
#     token=os.environ["HF_TOKEN"]
# )


# Не публикуйте свой токен и не добавляйте его в открытые репозитории.


In [3]:
import torch
import re
import torchaudio
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Dataset, Audio as HFAudio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    SpeechT5Processor,
    SpeechT5ForTextToSpeech,
    SpeechT5HifiGan,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from jiwer import wer, process_words

import matplotlib.pyplot as plt
from IPython.display import Audio, display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Устройство: cuda
GPU: Tesla T4


###Отберите 600 примеров из датасета - 0,5 балла

1. Проитерируйтесь по датасету `svarah`.

2. Отфильтруйте аудиофрагменты по длительности: пропускайте примеры,которые превышают MAX_DURATION.

3. Для каждого подходящего примера извлеките аудиомассив, частоту дискретизации и транскрипцию:

   - аудиомассив (`sample["audio_filepath"]["array"]`);
   - частота дискретизации (sampling_rate);
   - транскрипция.

4. Если частота дискретизации не равна `16000`, приведите аудио к `16 kHz` с помощью `torchaudio`.

5. Преобразуйте аудиомассив в формат `np.float32`.

6. Добавьте обработанный пример в список `collected` в следующем формате:

   ```python
   {
       "audio": {
           "array": аудиомассив,
           "sampling_rate": 16000
       },
       "sentence": текстовая_расшифровка
   }
7. Когда количество примеров достигнет TARGET(600), остановите процесс

In [ ]:
#streaming=True здесь нужен, чтобы не скачивать весь датасет Svarah целиком на диск, а читать примеры постепенно, “потоком”.
svarah = load_dataset("ai4bharat/Svarah", split="test", streaming=True)

# В Svarah всего ~6600 примеров, мы берем 600, так как этого должно хватить для заметных результатов
TARGET = 600

# Мы не берём в обработку аудио длиннее 20 секунд, потому что модель Whisper всё равно имеет ограничение по длине
# входного аудио и обычно обрабатывает его кусками примерно до 30 секунд.
MAX_DURATION = 20.0

collected = []
  #ВАШ КОД


### Создание объекта Dataset - 0,5 балла

На этом шаге вам нужно преобразовать список `collected` в объект `Dataset` [(можно посмотреть подробнее здесь)](https://huggingface.co/docs/datasets/loading) из библиотеки Hugging Face `datasets`.

1. Извлеките из списка `collected` отдельно:

   * аудиомассивы;
   * текстовые транскрипции.

2. Создайте датасет `train_real` с помощью `Dataset.from_dict`.

3. При создании датасета явно укажите структуру данных через `Features`:

   * колонка `"audio"` должна иметь тип `HFAudio(sampling_rate=16000)`;
   * колонка `"sentence"` должна иметь тип `Value("string")`.

4. Для колонки `"audio"` сохраните каждый пример в формате:

   ```python
   {
       "array": аудиомассив,
       "sampling_rate": 16000
   }
   ```

5. После создания датасета выведите количество записей в `train_real`.

В результате у вас должен получиться датасет `train_real`, где аудио уже приведено к формату Hugging Face Audio с частотой дискретизации `16 kHz`, а текстовые расшифровки хранятся в колонке `"sentence"`.


In [ ]:
from datasets import Features, Value

# Собираем через from_dict с явными Features.
# Указываем audio как HFAudio СРАЗУ при создании, тогда datasets сам кодирует
# Извлекаем аудиомассивы из collected
audio_col    = [] #ВАШ КОД
sentence_col = [] #ВАШ КОД

train_real = Dataset.from_dict()
    #ВАШ КОД
  ,
    features=Features( #ВАШ КОД),
)


Стриминг данных — довольно долгий шаг. Чтобы не повторять его при каждом переподключении Colab, сохраните собранную выборку на Google Drive ячейкой ниже. При следующем запуске она загрузится за секунды.

In [ ]:
# (Опционально) Кэширование train_real на Google Drive
# Запустите ОДИН раз чтобы примонтировать Drive:
# from google.colab import drive
# drive.mount('/content/drive')

import os
CACHE_PATH = ""   # путь на вашем Drive

# Сохранить (после того как train_real собран стримингом):
# train_real.save_to_disk(CACHE_PATH)
# print("Сохранено на Drive")

# Загрузить в следующий раз (вместо стриминга):
# from datasets import load_from_disk
# train_real = load_from_disk(CACHE_PATH)
# print(f"Загружено из кэша: {len(train_real)} записей")

In [ ]:
print("Примеры из обучающей выборки:\n")
for i in range(5):
    sample = train_real[i]
    print(f"[{i}] Текст: {sample['sentence']}")
    display(Audio(sample["audio"]["array"], rate=sample["audio"]["sampling_rate"]))
    print()

##2. Генерация синтетического набора данных - 3,5 балла

Реальных записей call-центра с индийским акцентом в открытых датасетах нет, поэтому генерируем сами через SpeechT5 с реальным Indian speaker embedding.
Нам нужен тестовый набор, где будут реалистичные для call центра фразы.


Что необходимо сделать:
1.   Необходимо написать функцию `extract_speaker_embedding` и собрать несколько голосов из датасета, которые позволят сгенерировать речь с акцентом - 1 балл
3. Придумать 20 фраз для генерации с усложнением - 1 балл
4. Сгенерировать эти фразы с помощью TTS и сохранить как HF Dataset - 1 балл
5. Проанализировать результаты генерации - 0,5 балла



In [ ]:
#Загружаем SpeechT5 и vocoder...
tts_processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
tts_model     = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to(device)
vocoder       = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)
tts_model.eval()
vocoder.eval()

In [ ]:
# Загружаем speaker encoder для извлечения x-vectors
# x-vector — стандартный 512-мерный embedding идентичности диктора
# Используем модель из SpeechBrain, обученную на VoxCeleb

# С SpeechBrain 1.0 модуль переименован: speechbrain.pretrained → speechbrain.inference
try:
    from speechbrain.inference import EncoderClassifier
except ImportError:
    from speechbrain.pretrained import EncoderClassifier

spk_encoder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-xvect-voxceleb",
    run_opts={"device": str(device)},
    savedir="./pretrained_models/spkrec-xvect"
)


###2.1 Извлечение speaker_embedding -0,5 балла

In [ ]:
def extract_speaker_embedding(audio_array, sr=16000):
    """
    Извлекает 512-мерный x-vector из аудио.

    Pipeline:
      аудио → ресемплинг до 16kHz → speaker encoder → embedding (1,1,512) → squeeze → (512,)

    Args:
        audio_array: numpy waveform
        sr:          sample rate входного аудио

    Returns:
        torch.Tensor формы (512,) на CPU
    """
    # Преобразуйте numpy audio_array в torch.Tensor типа float
    audio_tensor =   # ВАШ КОД

    if sr != 16000:
        # Сделайте resample с помощью torchaudio
        audio_tensor =   # ВАШ КОД

    # Добавьте размерность батча и перенесите тензор на устройство
    #Должно быть (1, n_samples)
    audio_tensor =   # ВАШ КОД

    # Отключите вычисление градиентов
    with    :   # ВАШ КОД (допишите условие)
        # Извлеките speaker embedding с помощью spk_encoder.encode_batch(...)
        embedding =   # ВАШ КОД

    # Уберите лишние размерности (с (1, 1, 512) на (512,)) и перенесите результат на CPU
    embedding =   # ВАШ КОД
    return embedding


###Подбор speaker_embedding - 0,5 балла

In [ ]:
indices = list(range(len(train_real)))
random.seed(48)            # фиксируем seed для воспроизводимости
random.shuffle(indices)
# Извлекаем несколько эмбеддингов от разных дикторов для разнообразия
#Здесь стоит поэкспериментировать с seed-ом и подобрать , которые будут искажать речь достаточно сильно
speaker_embeddings = []

for idx in indices:
    sample = train_real[idx]
    # Посчитайте длительность записи в секундах
    # (длина массива делёная на sampling_rate)
    duration =   # ВАШ КОД

    # Берём только записи длиннее 5 секунд
    if duration >= 5.0:
        # Извлеките embedding с помощью extract_speaker_embedding
        emb =   # ВАШ КОД

        # Приведите к форме (1, 512), перенесите на device и добавьте в список
        speaker_embeddings.append(  )   # ВАШ КОД

    # Останавливаемся, когда набрали 15 голосов
    if len(speaker_embeddings) >= 15:
        break

print(f"Собрано голосов: {len(speaker_embeddings)}")

###Создаем скрипты для озвучивания - 1 балл

In [14]:
# 20-30 типичных фраз, которые встречаются в реальной работе агента поддержки
# Специально включите сложные элементы:
#   - числа ("twenty four hours", "fifty dollars")
#   - сложные имена собственные ("McDougle")
#   - соответствующая лексика ("dispatched", "escalated", "subscription")
#   - вежливые формулы ("apologize for the inconvenience")

CALL_CENTER_SCRIPTS = [ "I have updated your delivery address to four two five Oak Street.",
    "Your package was shipped via FedEx and is currently in transit.",
    "Please log in to your account on our website to confirm.",
    "Please note your support ticket number is seven three nine zero.",
    "Your order reference is A B 1jkfgjf.",
]
#ВАШИ ПРИМЕРЫ



###Синтез речи - 1 балл

In [ ]:
def synthesize_speech(text, speaker_embedding):
    """
    Синтезирует речь по тексту с помощью SpeechT5 и HiFi-GAN vocoder.

    Pipeline:
      text → SpeechT5 tokenizer → SpeechT5 model → mel-spectrogram
      → HiFi-GAN vocoder → waveform

    Args:
        text: входная строка для синтеза речи
        speaker_embedding: speaker embedding формы (1, 512) на device

    Returns:
        numpy array — аудиосигнал с частотой 16 kHz
    """

    # Токенизируйте входной текст с помощью tts_processor.
    # Не забудьте указать return_tensors="pt" и перенести результат на device.
    inputs = # ВАШ КОД

    # Отключаем вычисление градиентов, так как на этом шаге модель используется только для инференса.
    with torch.no_grad():

        # Сгенерируйте речь с помощью tts_model.generate_speech.
        # Передайте:
        # - input_ids из inputs;
        # - speaker_embedding;
        # - vocoder.
        speech = # ВАШ КОД

    # Перенесите результат на CPU и преобразуйте в numpy array.
    speech = # ВАШ КОД

    return speech

In [ ]:
# Сгенерируйте 20 тестовых звонков с разными speaker_embededing (можете их выбирать рандомно
# или прям выбрать несколько, которые позволяют получить наиболее выраженный акцент)
test_audios = []
for i, text in enumerate(CALL_CENTER_SCRIPTS):
   # ВАШ КОД

# Собираем HuggingFace Dataset
# cast_column нужен чтобы datasets корректно работал с audio (для resampling и т.д.)
test_set = Dataset.from_dict(
    {
        # Сформируйте колонку audio.
        # Для каждого аудиомассива из test_audios создайте словарь c "array" в формате np.float32,
        # и "sampling_rate"
        "audio": # ВАШ КОД,

        # В колонку sentence добавьте тексты,
        # по которым синтезировалась речь
        "sentence": # ВАШ КОД,
    },

    # Явно задаем типы колонок датасета
    features=Features({
        # audio — аудиоколонка Hugging Face с частотой 16 kHz
        # sentence — строковая колонка
    }),
)

# При необходимости ещё раз приведите колонку audio к типу HFAudio
# test_set = test_set.cast_column(
#     "audio",
#     HFAudio(sampling_rate=16000)
# )


###Анализ результатов генерации - 0,5 балла

In [ ]:
# Прослушайте сгенерированные звонки
for i in range(len(test_set)):
    sample = test_set[i]
    print(f"[{i}] {sample['sentence']}")
    display(Audio(sample["audio"]["array"], rate=16000))
    print()


### Вопрос 1. Качество синтеза

Как звучат сгенерированные звонки по сравнению с реальными записями из Svarah?

Обратите внимание на:

- чистоту звука;
- наличие или отсутствие фонового шума;
- естественность речи и пауз;
- эмоции и интонацию.

### Вопрос 2. Акцент и разнообразие голосов

Слышен ли в сгенерированных звонках индийский акцент?

Также сравните голоса, сгенерированные с разными `speaker embeddings`:

- звучат ли они по-разному;
- насколько сильно меняется голос;
- похоже ли это на разных людей или различия небольшие.

---
## Часть 3. Whisper baseline - 1,5 балла

Сначала измерим WER исходной модели Whisper на тестовых звонках. На этом этапе модель не дообучается, поэтому результат показывает качество «как есть». Это будет наш baseline - точка отсчёта, с которой мы потом сравним модель после fine-tuning.

Что необходимо сделать:
1.   Дописать функцию для нормализации текстов (транскрипции Whisper и придуманных фраз), если необходимо
2.   Дописать функцию `transcribe batch` -1 балл
3.   Проанализировать результаты - 0,5 балла





In [17]:
# Загружаем whisper-small (244M параметров). Если не хватает памяти, попробуйте tiny
# WhisperProcessor включает:
#   feature_extractor: аудио → лог-мел спектрограмма (80 каналов × 3000 фреймов = 30 сек)
#   tokenizer:         текст → токены и обратно


whisper_model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(whisper_model_name)
baseline_model = WhisperForConditionalGeneration.from_pretrained(whisper_model_name).to(device)
baseline_model.eval()

print(f"Whisper-small: {sum(p.numel() for p in baseline_model.parameters()) / 1e6:.0f}M параметров")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

Whisper-small: 242M параметров


In [18]:
def normalize_text(text):
    """
    Нормализация для честного расчёта WER.

    Whisper выдаёт текст с пунктуацией и заглавными буквами, а наши эталоны —
    тоже с пунктуацией, но не обязательно совпадающей. Без нормализации
    "support." vs "support" засчитывается как ошибка, что искусственно завышает WER.

    Стандартная практика ASR-оценки - нормализация текста
    """
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)   # убираем пунктуацию
    text = re.sub(r"\s+", " ", text)        # убираем лишние пробелы
    return text

In [27]:
def transcribe_batch(model, processor, samples, target_sr=16000, verbose=False):
    """
    Транскрибирует список аудио-сэмплов моделью Whisper.

    Args:
        model:     WhisperForConditionalGeneration (на device)
        processor: WhisperProcessor
        samples:   список dict с полями 'audio' (array, sampling_rate) и 'sentence'
        target_sr: всегда 16000 для Whisper
        verbose:   печатать прогресс

    Returns:
        predictions: список нормализованных строк
        references:  список нормализованных эталонных строк
    """
    predictions = []
    references  = []

    for i, sample in enumerate(samples):
        audio_array = sample["audio"]["array"]
        sr = sample["audio"]["sampling_rate"]

        # Ресемплим если нужно
        if sr != target_sr:
            audio_tensor = torch.tensor(audio_array).float().unsqueeze(0)
            audio_array = torchaudio.functional.resample(audio_tensor, sr, target_sr).squeeze().numpy()

        # Аудио → мел-спектрограмма (1, 80, 3000)
        inputs = processor(
            audio_array,
            sampling_rate=target_sr,
            return_tensors="pt"
        ).input_features.to(device)

        # Генерация транскрипции (greedy decoding)
        # max_new_tokens=200 чтобы не уходить в бесконечность на сложных примерах
        with torch.no_grad():
            predicted_ids = model.generate(input_features=inputs,
        max_new_tokens=200
    )

        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        # Нормализуем оба текста одинаково для честного WER
        predictions.append(normalize_text(transcription))
        references.append(normalize_text(sample["sentence"]))

        if verbose:
            print(f"  [{i+1}/{len(samples)}]")

    return predictions, references


In [21]:
def compute_wer(predictions, references):
    """
    WER = (substitutions + deletions + insertions) / число слов в reference.

    Интерпретация:
      0.00 — идеально
      0.10 — 1 ошибка на 10 слов (норма для непростых данных)
      0.30+ — модель часто ошибается
      1.00+ — модель хуже чем "пустая строка" (бывает при многих insertions)
    """
    return wer(references, predictions)

In [ ]:
# Прогоняем baseline на наших 20 тестовых звонках
print("Baseline evaluation на тестовых звонках...")
test_samples = [test_set[i] for i in range(len(test_set))]

baseline_preds, baseline_refs = transcribe_batch(baseline_model, processor, test_samples)
baseline_wer_score = compute_wer(baseline_preds, baseline_refs)

print(f"\n=== Baseline WER: {baseline_wer_score:.3f} ({baseline_wer_score*100:.1f}%) ===\n")

# Показываем все 20 примеров, чтобы можно было выбрать 5 для ручного анализа
for i, (ref, pred) in enumerate(zip(baseline_refs, baseline_preds)):
    is_exact_match = ref == pred
    status = "✓" if is_exact_match else "✗"

    print(f"\n[{i}] {status}")
    print(f"  REF:  {ref}")
    print(f"  PRED: {pred}")

###Анализ ошибок

**Вопрос 1. Общая картина.**
Какой WER показала baseline-модель? Оцените, насколько хорошо справляется модель с распознаванием сгенерированной речи.

**Вопрос 2. Разбор конкретных ошибок.**
Выберите 3 примера, где модель ошиблась. Для каждого выпишите REF и PRED и определите тип ошибки:
- **substitution** — слово заменено на другое
- **deletion** — слово пропущено
- **insertion** — лишнее слово

Какой тип преобладает?



---
## Часть 4. Файнтюнинг - 4 балла

Полный fine-tuning `whisper-small` слишком тяжёлый для Colab T4: у модели около **244 млн параметров**, и во время обучения нужно хранить не только веса модели, но и градиенты, состояния оптимизатора и промежуточные активации. Из-за этого обучение всех параметров может не поместиться в память GPU.

По этой причине мы используем **LoRA** (*Low-Rank Adaptation*) — параметрически эффективный способ дообучения больших моделей.


In [20]:
import sys
sys.modules["torchao"] = None

###Дописать DataCollator - 1 балл


Во время обучения модель получает данные не по одному примеру, а батчами. Однако аудио и текстовые транскрипции имеют разную длину, поэтому их нужно привести к единому формату.

Для этого используется **Data Collator**. Он выполняет несколько действий:

1. Собирает аудио из всех примеров батча и преобразует его в признаки (`input_features`), которые ожидает Whisper.
2. Токенизирует текстовые транскрипции.
3. Добавляет padding к коротким текстам, чтобы все последовательности в батче имели одинаковую длину.
4. Заменяет padding-позиции на `-100`, чтобы функция потерь игнорировала их при обучении.
5. Возвращает готовый батч, который можно передать модели.

Ваша задача — заполнить пропуски и понять, как происходит подготовка батча для обучения Whisper.

Что нужно сделать:



*   Преобразовать аудио в признаки, которые можно подать Whisper
*   Дописать параметры для токенизатора
*   Предотвратить дублирование токенов в начале




In [ ]:
# Whisper — seq2seq модель:
#   - Encoder получает аудио → лог-мел спектрограмма фиксированного размера (80, 3000)
#   - Decoder получает текстовые токены переменной длины → нужен padding в батче
#

#dataclass — это специальный декоратор из стандартной библиотеки Python, который автоматически создаёт конструктор класса
# и некоторые служебные методы. В этой работе @dataclass используется только для того, чтобы сделать код короче и удобнее.
# Никакой дополнительной логики для подготовки батчей он не добавляет.
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:

        # Шаг 1: аудио → мел-спектрограмма (batch, 80, 3000)
        # Преобразуйте список аудиомассивов в признаки, которые ожидает Whisper.Укажите:
        # - audio_arrays как входные данные;
        # - sampling_rate=16000, так как Whisper обучался на аудио 16 kHz;
        # - return_tensors="pt", чтобы результат был представлен в виде PyTorch-тензоров.
        # Из результата сохраните только поле input_features.
        audio_arrays = [f["audio"]["array"] for f in features]
        input_features = self.processor... #ВАШ КОД

        # Шаг 2: текст → токены с паддингом
        # Токенизатор возвращает input_ids и attention_mask (1 = настоящий токен, 0 = padding)
        # Токенизируйте все транскрипции из батча. Для этого извлеките поле "sentence" из каждого примера в features
        # Включите padding, truncation=True,установите максимальную длину последовательности 448
        #Cделайте так, чтобы токенизатор вернул тензоры

        label_batch = self.processor.tokenizer(
        #ВАШ КОД

        )

        # Шаг 3: маскируем padding значением -100.

        # у Whisper токен padding совпадает с токеном конца реплики (eos), и по id их не различить
        # Маскируем по attention_mask, а НЕ по pad_token_id
        # Логика такая: заменяем все 0 в attention_mask на - 100, которые и являются
        # padding, чтобы модель во время обучения не выучила это как токен конца реплики
        # ne() - not equal
        labels = label_batch["input_ids"].masked_fill(
            label_batch["attention_mask"].ne(1), -100
        )

        # Шаг 4: срезаем дублирующий стартовый токен.
        # Токенизатор добавляет <|startoftranscript|> в начало, и модель добавляет его же
        # при обучении, без среза получился бы дубль.
        # Получаем id токена <|startoftranscript|>, Проверяем, что первый токен во ВСЕХ примерах батча
        #действительно является стартовым токеном. Если это так, удаляем первый столбец из labels.

        sot_id = self.processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
        #ВАШ КОД

        return {"input_features": input_features, "labels": labels}


# Быстрая проверка, что collator собирает батч без ошибок
collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
batch = collator([test_set[0], test_set[1]])
print(f"input_features: {batch['input_features'].shape}")  # (2, 80, 3000)
print(f"labels:         {batch['labels'].shape}")

###Работа с моделью - 2 балла

Здесь необходимо прописать цикл обучения модели и проанализировать лоссы по время обучения

Реализация цикла со всеми параметрами - 1,5 балла

Анализ результатов (лоссов во время обучения) - 0,5 балла

In [24]:
def build_lora_model(base_model_name="openai/whisper-small"):
    """
    Загружает Whisper и оборачивает его в LoRA.

    Конфигурация LoRA:
        r=8 — ранг разложения (больше = больше параметров и ёмкости)
        lora_alpha=16 — масштабирование (обычно 2*r)
        target_modules — q_proj и v_proj в attention (стандарт для трансформеров)
        lora_dropout=0.05 — лёгкая регуляризация

    Returns:
        PeftModel — Whisper с замороженной базой и обучаемыми LoRA-адаптерами
    """
    model = WhisperForConditionalGeneration.from_pretrained(base_model_name)

    # Специфично для Whisper + LoRA:
    # - forced_decoder_ids=None: при обучении не форсим конкретный язык, модель учится по labels
    # - use_cache=False: kv-cache несовместим с gradient checkpointing
    model.config.forced_decoder_ids = None
    model.config.suppress_tokens    = []
    model.config.use_cache          = False

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()  # покажет долю обучаемых параметров
    return model.to(device)


# Создаём модель для обучения
lora_model = build_lora_model()

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

trainable params: 884,736 || all params: 242,619,648 || trainable%: 0.3647


In [ ]:

#
# Основные шаги обучения:
#
# 1. Создать DataLoader.
# 2. Создать optimizer.
# 3. Перевести модель в режим train().
# 4. Для каждой эпохи пройтись по батчам.
# 5. Передать batch в модель.
# 6. Получить loss.
# 7. Сделать backward().
# 8. Обновить параметры optimizer.step().
# 9. Очистить градиенты optimizer.zero_grad().
# 10. Сохранять значения loss для графика.

from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

# Параметры обучения
NUM_EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
GRAD_ACCUM_STEPS = 2

# Создаем DataLoader для train_real.

train_loader = DataLoader(
    train_real,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
)

# Создаем optimizer.
# Для LoRA мы обучаем только параметры, у которых requires_grad=True.
optimizer = AdamW(
    #ВАШ КОД,
    lr=LEARNING_RATE,
)

# Переведите модель в режим обучения.
lora_model.train()

# Здесь будем хранить значения loss для графика.
loss_history = []


# Цикл по эпохам
for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")

    # tqdm показывает прогресс внутри эпохи
    progress_bar = tqdm(train_loader)

    # Обнуляем градиенты перед началом эпохи

    # Цикл по батчам
    for step, batch in enumerate(progress_bar):

        # Перенесите все тензоры батча на device.
        # batch содержит input_features и labels.
        batch = #ВАШ КОД

        # Передайте batch в модель.
        # Whisper сам посчитает loss, если передать labels.
        outputs = lora_model()#ВАШ КОД

        # Достаньте loss из outputs.
        loss = #ВАШ КОД

        # Делим loss на GRAD_ACCUM_STEPS,
        # потому что будем накапливать градиенты несколько шагов.
        loss = #ВАШ КОД

        # Считаем градиенты.
        loss.backward()

        # Обновляем параметры не на каждом шаге, а раз в GRAD_ACCUM_STEPS шагов.
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            optimizer...  #ВАШ КОД
            optimizer... #ВАШ КОД

        # Сохраняем настоящий loss для графика.
        loss_value = loss.item() * GRAD_ACCUM_STEPS
        loss_history.append(loss_value)

        # Показываем текущий loss в progress bar.
        progress_bar.set_postfix({"loss": loss_value})

print(f"\nОбучение завершено. Зафиксировано {len(loss_history)} точек loss.")

In [ ]:
# Кривая обучения
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(loss_history, color="steelblue", linewidth=2)
ax.set_xlabel("Шаг логирования (каждые 20 батчей)")
ax.set_ylabel("Training loss")
ax.set_title("Кривая обучения: LoRA Whisper-small на Indian English")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()

**Проанализируйте результаты дообучения** на основе сохраненной истории лоссов

###Подсчет WER после файнтьюнинга - 1 балла

Прогоните дообученную модель на тех же сгенерированных звонках и сравните WER бейзлайн модели и дообученной.
Перед инференсом переведите модель в режим оценивания с помощью model.eval(). Затем получите предсказания дообученной модели, посчитайте WER и сравните результат с WER baseline-модели.

Проанализируйте результаты и ответьте на **вопросы**:

1. Улучшилась ли модель после дообучения? На основании каких результатов вы делаете этот вывод?

2.  Есть ли примеры, где fine-tuned модель стала хуже baseline? Как вы думаете, почему это могло произойти?

3. Какие дополнительные эксперименты вы бы провели, чтобы убедиться, что fine-tuning действительно улучшил качество модели?






In [32]:
#это необходимо, чтобы функция transcribe_batch работала с lora_model, иначе код будет ломаться
# переносим suppress_tokens в правильное место
lora_model.generation_config.suppress_tokens = []

# убираем suppress_tokens из model.config
lora_model.config.suppress_tokens = None

# на всякий случай то же самое для forced_decoder_ids
lora_model.generation_config.forced_decoder_ids = None
lora_model.config.forced_decoder_ids = None

In [34]:
# ВАШ КОД

## Challenge: улучшение качества дообученной Whisper-модели

В этом задании вам нужно попробовать улучшить качество дообученной модели Whisper для распознавания английской речи с индийским акцентом.

### Задание

Примените **минимум 3 изменения** на любом этапе пайплайна:

* при подготовке данных;
* при препроцессинге аудио или текста;
* при настройке обучения;
* при настройке инференса;
* при постобработке предсказаний.

После этого снова обучите или дообучите модель, прогоните её на том же тестовом наборе и сравните WER с предыдущими результатами.

Важно: не все изменения обязаны улучшить качество. Некоторые преобразования могут ухудшить результат или почти не повлиять на него. Это нормально. Главное — провести эксперимент, зафиксировать изменения и проанализировать, как они повлияли на метрику.

### Примеры возможных изменений

Можно попробовать:

* увеличить количество обучающих примеров;
* изменить train/test split;
* добавить больше примеров с индийским акцентом;
* сбалансировать данные по длине аудио или типам фраз.
* унифицировать сокращения, например `I'm` → `I am`;
* learning rate;
* количество эпох;
* batch size;
* gradient accumulation steps;
* warmup steps;
* weight decay;
* параметры LoRA: `r`, `alpha`, `dropout`;
* `max_new_tokens`;
* `num_beams`;
* `language`;
* `task`;
* отключить или включить forced decoder ids;
* явно указать, что речь на английском языке.

### Критерии успешного выполнения

Challenge считается выполненным, если:

* вы применили минимум 3 изменения;
* заново оценили модель на том же тестовом наборе;
* посчитали WER;
* сравнили результат с baseline и предыдущей fine-tuned моделью;
* сделали короткий вывод.

Необязательно добиться улучшения WER. Важнее показать экспериментальный подход: сформулировать гипотезу, внести изменения, измерить результат и объяснить, что произошло.
